# CS4168 Group Project — Spotify Tracks Analysis

Dataset: "tracks2026.csv" 2,000 music tracks with audio features and metadata across 5 genres.

---
## 1. Exploratory Data Analysis (EDA)

### 1.1 Import Modules and Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler, FunctionTransformer, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold, cross_validate, cross_val_predict
from sklearn.metrics import (confusion_matrix, roc_curve, precision_recall_curve, auc,
                             accuracy_score, precision_score, recall_score, f1_score,
                             mean_squared_error, mean_absolute_error, r2_score)
from sklearn.inspection import permutation_importance
from sklearn import set_config
set_config(display='diagram')

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

print("All imports successful.")


In [ ]:
df = pd.read_csv("tracks2026.csv")
print(f"Dataset shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
df.head()

In [ ]:
df.tail()

### 1.2 Missing Value Analysis

In [ ]:
# Count missing values per column
missing = df.isna().sum()
missing_pct = (df.isna().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

print(f"\nTotal rows with at least one missing value: {df.isna().any(axis=1).sum()}")
print(f"Total rows in dataset: {len(df)}")

**Observations on missing values:**
- 5 columns have missing values: Popularity (40), Danceability (40), Energy (40), Loudness (39), and Tempo (40).
- All missing values are concentrated in the same 40 rows (2% of data).
- Since the missing data is a small proportion and appears to be missing at random, we will use median imputation via SimpleImputer inside our pipelines. Median is preferred over mean because several features contain outliers, as seen below.

### 1.3 Descriptive Statistics

In [ ]:
df.describe().round(4)

**Key observations from descriptive statistics:**
- Popularity ranges from 0 to 100 with a median of 45. The mean (≈40) being lower than the median indicates a left-leaning concentration, and 422 tracks have a popularity of exactly 0.
- Loudness has a maximum of 800,000 which is clearly erroneous, loudness in dB should be negative (typically -60 to 0). This is a single data entry error that must be handled.
- Instrumentalness has a mean near 0 and a 75th percentile near 0, the vast majority of tracks have little instrumental content with there being a few outliers.
- Duration_ms ranges from 60,000 (1 min) to 561,133 (≈9.4 min), with a right-skewed distribution.

### 1.3.1 Investigating the Loudness Outlier

In [ ]:
# Identify the erroneous loudness value
print("Rows with loudness > 0 (should not exist — loudness is measured in negative dB):")
print(df[df['loudness'] > 0][['track_id', 'loudness', 'track_genre']])

print(f"\nLoudness statistics (excluding the outlier):")
print(df[df['loudness'] <= 0]['loudness'].describe())

# Replace with NaN — it will be imputed later in our pipelines
df.loc[df['loudness'] > 0, 'loudness'] = np.nan
print("\n✓ Replaced erroneous loudness value (800,000) with NaN for imputation.")

The loudness value of 800,000 for the track "39ujbBjTwwqUFySaCYDMMT" is clearly a data entry error. Normal loudness values in this dataset range from -21.09 to -0.08 dB. We replace it with NaN so it will be handled by imputation in our preprocessing pipelines.

### 1.4 Feature Distributions

In [ ]:
numeric_cols = ['popularity', 'duration_ms', 'danceability', 'energy', 'loudness',
                'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    df[col].hist(bins=50, ax=axes[i], edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Frequency')
axes[-1].set_visible(False)
plt.suptitle('Distribution of Numerical Features', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Quantify skewness
print("Skewness of numerical features:")
print("-" * 45)
for col in numeric_cols:
    skew = df[col].skew()
    label = "(HIGH)" if abs(skew) > 1 else "(moderate)" if abs(skew) > 0.5 else ""
    print(f"  {col:20s}: {skew:+.3f} {label}")

**Observations on distributions:**
- Heavily right-skewed features: Speechiness (+2.47), Instrumentalness (+5.45), Liveness (+2.47), Duration_ms (+1.04). 
- Moderately skewed: Loudness (-0.82), Acousticness (+0.74).
- Approximately symmetric: Popularity, Danceability, Energy, Valence, Tempo.
- Popularity has a notable spike at 0 (422 tracks), creating a bimodal-like shape. This is important, it means popularity is hard to predict as a continuous value (many tracks cluster at zero).

Preprocessing implication: For features with high skewness (Speechiness, Instrumentalness, Liveness, Duration_ms), we will apply log transformation. Since some of these contain zeros, we will use log1p (i.e., log(1 + x)) to avoid undefined values.

### 1.5 Outlier Analysis

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    df.boxplot(column=col, ax=axes[i])
    axes[i].set_title(col, fontsize=12, fontweight='bold')
axes[-1].set_visible(False)
plt.suptitle('Boxplots of Numerical Features (Outlier Detection)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Observations on outliers:**
- Outlier-heavy features: Duration_ms, Speechiness, Acousticness, Instrumentalness, Liveness all show significant outliers above the upper whisker.
- Minimal outliers: Danceability, Energy, Valence, Tempo, all have relatively tight distributions with few or no extreme values.

Preprocessing implication: For features with many outliers, we will use RobustScaler (which scales based on the IQR and is resistant to outliers) rather than StandardScaler.

### 1.6 Genre Analysis

In [ ]:
print("Genre distribution:")
print(df['track_genre'].value_counts())
print(f"\nExplicit tracks by genre (proportion):")
print(df.groupby('track_genre')['explicit'].mean().round(3).sort_values(ascending=False))

In [ ]:
features_to_compare = ['popularity', 'danceability', 'energy', 'loudness', 'speechiness',
                       'acousticness', 'valence', 'tempo', 'instrumentalness']

fig, axes = plt.subplots(3, 3, figsize=(20, 14))
axes = axes.flatten()
for i, col in enumerate(features_to_compare):
    df.boxplot(column=col, by='track_genre', ax=axes[i], rot=15)
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')
plt.suptitle('Feature Distributions by Genre', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Mean feature values per genre
print("Mean feature values by genre:")
print(df.groupby('track_genre')[features_to_compare].mean().round(4).to_string())

**Observations on genre differences:**
- Hip-hop stands out with the highest Danceability (0.73) and Speechiness (0.13), and the highest proportion of explicit tracks (33%).
- Synth-pop has the highest Energy (0.71) and Instrumentalness (0.075), and the lowest Speechiness (0.048).
- R&B has the highest Valence (0.63), suggesting more "positive-sounding" tracks.
- Pop has the highest average Popularity (46.2), while r-n-b and synth-pop are lowest (≈36).
- Genres overlap substantially on most features, which suggests that clustering based on audio features alone may not perfectly recover genre boundaries.

Implications for clustering: The overlap in feature distributions between genres suggests clustering will likely find partial structure but not clean genre separation. Features like Speechiness, Danceability, and Energy show the most differentiation between genres and may be the most useful for clustering.

### 1.7 Correlation Analysis

In [ ]:
corr_cols = ['popularity', 'duration_ms', 'danceability', 'energy', 'key', 'loudness',
             'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1, annot_kws={'size': 9})
ax.set_title('Correlation Matrix of Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Correlations with popularity
print("Correlations with popularity (sorted by absolute strength):")
print("-" * 50)
pop_corr = corr['popularity'].drop('popularity').abs().sort_values(ascending=False)
for feat in pop_corr.index:
    actual = corr.loc[feat, 'popularity']
    print(f"  {feat:20s}: {actual:+.4f}")

print("\nStrong inter-feature correlations (|r| > 0.4):")
print("-" * 50)
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        r = corr.iloc[i, j]
        if abs(r) > 0.4:
            print(f"  {corr.columns[i]:20s} vs {corr.columns[j]:20s}: {r:+.4f}")


**Observations on correlations:**

**Correlations with popularity (target variable):**
- No audio feature has a strong linear correlation with popularity. The strongest is Valence at just -0.10. This is a critical finding: it suggests that popularity is not well-explained by audio features alone. Factors like artist fame, marketing, and playlist placement likely matter far more.
- This weak linear relationship means that regression on the raw popularity score will be challenging, while classification (above/below median) may perform better since it simplifies the problem to a binary decision.

**Inter-feature correlations:**
- Energy & Loudness: +0.63 (strong positive: louder tracks tend to be more energetic)
- Energy & Acousticness: -0.59 (strong negative: acoustic tracks tend to have lower energy)

**Implications for modelling:**
- Since no single feature strongly predicts popularity, non-linear models (e.g., Random Forest) may outperform linear models by capturing complex feature interactions.
- The weak correlations also suggest that track_genre (as a categorical feature, not shown in this numeric correlation matrix) may be one of the more useful predictors, given the genre-level differences in mean popularity observed in Section 1.6.

### 1.8 Target Variable Analysis — Popularity

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution with median line
df['popularity'].hist(bins=50, ax=axes[0], edgecolor='black', alpha=0.7, color='steelblue')
median_val = df['popularity'].median()
axes[0].axvline(median_val, color='red', linestyle='--', linewidth=2, label=f'Median = {median_val}')
axes[0].set_title('Popularity Distribution', fontweight='bold')
axes[0].set_xlabel('Popularity')
axes[0].legend()

# Binary split preview
pop_binary = (df['popularity'] > median_val).astype(int)
pop_binary.value_counts().sort_index().plot(kind='bar', ax=axes[1], color=['coral', 'steelblue'], edgecolor='black')
axes[1].set_title(f'Binary Popularity Split (median={median_val:.0f})', fontweight='bold')
axes[1].set_xticklabels(['≤ median (0)', '> median (1)'], rotation=0)
axes[1].set_ylabel('Count')

# By genre
df.boxplot(column='popularity', by='track_genre', ax=axes[2], rot=15)
axes[2].set_title('Popularity by Genre', fontweight='bold')
axes[2].set_xlabel('')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f"Tracks with popularity = 0: {(df['popularity'] == 0).sum()}")
print(f"Tracks with popularity ≤ 5: {(df['popularity'] <= 5).sum()}")
print(f"\nBinary target class balance (after dropping NaN):")
print(pop_binary.dropna().value_counts())


**Observations on popularity as a target:**
- Popularity has a bimodal distribution, a large spike at 0 (422 tracks, ~21%) and a broader spread from 20–100.
- The binary split at the median (45) produces a well-balanced target (994 vs 966), which is ideal for classification and means we do not need oversampling or class weighting.
- Popularity varies somewhat by genre: pop has the highest median, while r-n-b and synth-pop are lower.
- The concentration at 0 will make regression particularly difficult, since a large cluster of tracks are indistinguishable at the floor value. This already suggests classification will likely outperform regression.